# Protocolo OPeNDAP DAP



```{note}
**¿Por qué es importante entender el tipo de datos cubierto por cada protocolo?** Los servidores OPeNDAP que implementan `DAP2` y `DAP4` están escritos en `C/C++` y `Java`, lo que significa que las personas usuarias de PyDAP DEBEN conocer los tipos atómicos con los que estos servidores son compatibles.
```

En términos generales hay `dos modelos de datos DAP`: [DAP2](https://zenodo.org/records/10794666) y [DAP4](https://opendap.github.io/dap4-specification/DAP4.html). Desde la perspectiva del cliente, estos dos tienen pequeñas diferencias, así que abajo proporcionamos una breve descripción de ambos.




El modelo de datos DAP4 está cerca de ser un _superconjunto_ del modelo DAP2 antiguo, excepto que `Grids` ya no forman parte del modelo de datos DAP4 (ver [Figura 1](Figure1)). Sin embargo, como el tipo de datos `Grid` es un objeto DAP2 (a diferencia de un objeto `netCDF/HDF`), un _objeto Grid en DAP2 PUEDE representarse en DAP4_. Esto significa que cualquier dataset representado por el modelo de datos DAP2 (y, digamos, disponible mediante un servidor TDS/Hyrax) puede representarse por completo en DAP4. Sin embargo, los tipos y objetos DAP4 (como `Groups` e `Int64`) servidos por un servidor remoto Hyrax/TDS no se pueden representar en DAP2, y pydap recibirá respuestas DAP a las que les faltan esos tipos de datos.

Para aprender más sobre la especificación DAP4, consulta la [documentación oficial de DAP4](https://opendap.github.io/dap4-specification/DAP4.html), escrita conjuntamente por Unidata y OPeNDAP, Inc en 2016.


| ![Figure1](/images/DAP4vsDAP2.png) |
|:--:|
| *Figura 1. Comparación entre modelos de datos y respuestas DAP2 y DAP4.* |


PyDAP busca cubrir ampliamente todos los modelos de datos DAP2 y DAP4, y por ello cubre los siguientes objetos DAP:

* Groups
* Gridded Arrays.
* Sequences (datos tabulares)
* Structures.

* Todos los tipos atómicos excepto `Opaque` y `Enum` (todos los demás tipos pueden representarse con datos de arreglos numpy).

Al abrir una URL remota, `PyDAP` creará un objeto `Dataset` que actúa como el directorio `root`. El `Dataset` de `PyDAP` puede contener múltiples tipos de datos, que pueden estar `anidados`. Como `PyDAP` se acerca al soporte completo del modelo `DAP4`, soporta `Groups` y `Groups` anidados, que a su vez pueden contener otros tipos de datos y otros objetos `PyDAP` anidados nombrados arriba.

## Groups

Los Groups son en gran medida una característica de los modelos de formato de archivo `HDF5/NetCDF4`. Muchos datasets remotos (por ejemplo, de NASA disponibles mediante servidores de datos Hyrax) pueden contener uno o más `Groups`, algunos de ellos anidados. Debido a la compleja especificación del modelo de datos HDF, la especificación `DAP4` sigue de cerca el modelo `netCDF4`. Esto significa que no hay `Groups` cíclicos. En su lugar, siempre hay una raíz, y cada `Group` tiene un único `Group` padre.

Para leer datos desde un servidor DAP4 remoto, DEBES establecer `open_url(..., protocol='dap4')`.


Considera el siguiente ejemplo:


In [ ]:
from pydap.client import open_url
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
dataset = open_url('http://test.opendap.org:8080/opendap/atlas03/ATL03_20181228015957_13810110_003_01.h5', protocol='dap4')
dataset

`print(dataset)` solo devuelve los elementos dentro del directorio raíz. En `DAP4`, se puede navegar el dataset como si fuera un sistema de archivos POSIX e inspeccionar las variables dentro de un grupo (anidado). Por ejemplo:


In [ ]:
dataset['gt1r']

In [ ]:
dataset['/gt1r/bckgrd_atlas'].tree()

Observamos la variable `bckgrd_int_height` y sus `dimensions`:


In [ ]:
dataset['/gt1r/bckgrd_atlas/bckgrd_int_height'].attributes

In [ ]:
dataset['/gt1r/bckgrd_atlas/bckgrd_int_height'].dimensions # dimensions of the variable

In [ ]:
dataset['/gt1r/bckgrd_atlas'].dimensions # Dimension of the Group

```{note}
La `dimension` `delta_time` está definida a nivel de `Group`, con un nombre relativo al `Group:/gt1r/bckgrd_atlas` (anidado). Pero al inspeccionar un arreglo de variable que lista `delta_time` como su dimensión, la lista con su Fully Qualifying Name (FQN).
```

Los `Fully Qualifying Names` son una característica importante del modelo de datos DAP4, evitando colisiones de nombres entre Groups. Para leer más sobre Qualifying Names, te remitimos a la [especificación oficial DAP4](https://opendap.github.io/dap4-specification/DAP4.html#_fully_qualified_names).


Intentar leer este dataset desde un servidor DAP4 puede producir un error.


In [ ]:
import requests
try:
    open_url('http://test.opendap.org:8080/opendap/atlas03/ATL03_20181228015957_13810110_003_01.h5', protocol='dap2')
except:
    print("Your request was for a response that uses the DAP2 data model. This dataset contains variables whose data type is not compatible with that data model, causing this request to FAIL. To access this dataset ask for the DAP4 binary response encoding.")

En el caso anterior, una variable es de tipo `Int64` (atómico), lo que causa el error. Sin embargo, si en cambio el atributo es de un tipo no soportado por el modelo DAP2, el servidor no lanzará un `HTTPError`, y `PyDAP` descargará un dataset `incomplete` (con atributos faltantes).


## Arrays / Grids

Empecemos accediendo a un `Gridded Array`, es decir, datos almacenados como un arreglo multidimensional regular. Aquí hay un ejemplo simple donde accedemos a la climatología [COADS](https://icoads.noaa.gov/) desde el `servidor de pruebas OPeNDAP` oficial:


In [ ]:
dataset = open_url('http://test.opendap.org/dap/data/nc/coads_climatology.nc', protocol='dap4') # dap2 is the default
dataset

Aquí usamos la función `pydap.client.open_url` para abrir una URL `OPeNDAP` que especifica la ubicación del dataset. Cuando accedemos al dataset remoto, la función devuelve un objeto `DatasetType`, que es un diccionario elegante que almacena otras variables. Podemos comprobar los nombres de las variables almacenadas como haríamos con un diccionario de Python:


Otra forma útil de inspeccionar el dataset es el método `.tree()`, que proporciona una inspección detallada de tu dataset.


In [ ]:
dataset.tree()

```{note}
Todavía no se ha descargado ningún dato.
```

Trabajemos con la variable `SST`; podemos referenciarla de forma "perezosa" usando la sintaxis usual de diccionario `dataset['SST']`, o `dataset.SST`:


In [ ]:
dataset['/SST'].attributes

In [ ]:
dataset['/SST'].dimensions

In [ ]:
%%time
sst = dataset['SST'][0, 45:80, 45:125]
print('Type of object: ', type(sst))

### Enfoque antiguo de `DAP2`

Abajo accedemos al mismo dataset en el mismo servidor y solicitamos la respuesta DAP2.


In [ ]:
dataset = open_url('http://test.opendap.org/dap/data/nc/coads_climatology.nc', output_grid=True) # dap2 is the default
dataset

In [ ]:
dataset.tree()

In [ ]:
sst = dataset['SST'] # or dataset.SST
sst

Observa que la variable es de tipo `GridType`, un arreglo multidimensional con ejes específicos que definen cada una de sus dimensiones:


In [ ]:
sst.dimensions

In [ ]:
sst.shape

In [ ]:
len(sst)

```{note}
`len` y `shape` de `sst` difieren. La diferencia entre ellos es una propiedad de `GridType`: en este caso `sst` incluye consigo los `coordinate axes` necesarios para ser completamente "autodescriptivo". Este comportamiento se hizo para imitar el modelo `netCDF`, donde cada archivo (en este caso cada `gridded array`) es "autodescriptivo" y por lo tanto contiene toda la información relevante para procesarlo. La vista `tree` del dataset anterior ilustra aún más el punto, con múltiples copias de datos de coordenadas en el dataset. Cada vez que descargas un `GridType`, también descargas los arreglos de dimensión usados para describirlo por completo.

```


Es posible especificar que NO se descarguen los ejes de coordenadas de una variable `GridType` al abrir una URL de la siguiente manera:


In [ ]:
dataset = open_url("http://test.opendap.org/dap/data/nc/coads_climatology.nc")
dataset

NO se han descargado datos en memoria todavía, pero cuando descargues un arreglo `GridType`, los ejes de coordenadas no se descargarán. Este workaround importante es particularmente útil para acelerar flujos de trabajo y no sobrecargar servidores `OPeNDAP` antiguos que podrían quedarse sin memoria al intentar recuperar tanto los datos como los ejes de coordenadas de una variable.


```{note}
`GridType` es una especificación del modelo `DAP2` y se eliminó en el mucho más nuevo `DAP4`. No obstante, los servidores `OPeNDAP DAP4` soportan `DAP2`. Actualmente, cuando `PyDAP` abre una URL remota, si no se especifica `protocol`, `PyDAP` asume por defecto la especificación del modelo DAP2. Esto puede cambiar en el futuro.
```


En `PyDAP`, `BaseType` es una envoltura ligera de un `numpy.ndarray` con algunos de los mismos atributos básicos, como shape y nbytes. `Dap4BaseProxy` es un contenedor de datos vacío con todos los atributos del arreglo remoto, incluyendo:

- `name`
- `dtype`
- `shape`
- `slice`



Para `descargar` datos en memoria debes `rebanar` el arreglo. Esto es:


In [ ]:
%%time
sst = dataset['SST']['SST'][:]
print('Type of object: ', type(sst))

In [ ]:
type(sst.data)

```{note}
Una de las características de OPeNDAP es el `server-side subsetting`, que `PyDAP` puede aprovechar mediante el rebanado del arreglo al descargar datos. Por ejemplo, si solo te interesa una subregión localizada de todo el dominio (por ejemplo, un rango finito de latitudes y longitudes de la cobertura global), y conoces los índices que abarcan tu área de interés deseada.
```


In [ ]:
%%time
sst = dataset['SST'][0, 45:80, 45:125]
print('Type of object: ', type(sst))

In [ ]:
sst.shape

```{note}
El subconjunto ocurrió cerca de los datos gracias a la funcionalidad del servidor OPeNDAP, y solo se descargan los datos que solicitamos en el rebanado.
```

```{note}
Rebanar el arreglo antes de descargar casi siempre resultará en mejor rendimiento y aceleraciones. Esto puede ser significativo para datasets muy grandes y particularmente útil al automatizar flujos de trabajo que requieren descargar datos en memoria para procesamiento posterior.

```


## Datos tabulares


Veamos ahora datos tabulares, que `PyDAP` llama `Sequences` (típicamente asociadas con datos in situ secuenciales). Aquí consideramos otro archivo dentro del servidor de pruebas OPeNDAP por simplicidad. Puedes acceder al catálogo [AQUÍ](http://test.opendap.org/opendap/hyrax/).

```{note}
Los datos secuenciales consisten en uno o más registros de variables relacionadas, como mediciones simultáneas de temperatura y velocidad del viento, por ejemplo.
```


In [ ]:
ds = open_url("http://test.opendap.org/opendap/hyrax/data/ff/gsodock1.dat")
ds.tree()

In [ ]:
ds['URI_GSO-Dock']

### Datos in situ de ERDDAP


Considera un ejemplo más complejo. Aquí miramos datos del DAC de [glider](https://oceanservice.noaa.gov/facts/ocean-gliders.html) en el [Integrated Ocean Observing System](https://data.ioos.us/organization/glider-dac). Los datos se pueden acceder a través de un servidor OPeNDAP, así como del servidor [ERRDAP](https://gliders.ioos.us/erddap/index.html). En el ejemplo abajo demostramos cómo acceder a datos de glider de un estudio Deep-Pelagic Nekton frente al Golfo de México, con pydap mediante `ERRDAP`.

Miramos el `Dataset ID: Murphy-20150809T1355`. Su formulario de acceso `ERDDAP` se puede encontrar [AQUÍ](https://gliders.ioos.us/erddap/tabledap/Murphy-20150809T1355.html?trajectory%2Cwmo_id%2Cprofile_id%2Ctime%2Clatitude%2Clongitude%2Cdepth%2Cconductivity%2Cconductivity_qc%2Cdensity%2Cdensity_qc%2Cdepth_qc%2Cinstrument_ctd%2Clat_qc%2Clat_uv%2Clat_uv_qc%2Clon_qc%2Clon_uv%2Clon_uv_qc%2Cplatform_meta%2Cprecise_lat%2Cprecise_lon%2Cprecise_time%2Cpressure%2Cpressure_qc%2Cprofile_lat_qc%2Cprofile_lon_qc%2Cprofile_time_qc%2Cqartod_conductivity_flat_line_flag%2Cqartod_conductivity_gross_range_flag%2Cqartod_conductivity_primary_flag%2Cqartod_conductivity_rate_of_change_flag%2Cqartod_conductivity_spike_flag%2Cqartod_density_flat_line_flag%2Cqartod_density_gross_range_flag%2Cqartod_density_primary_flag%2Cqartod_density_rate_of_change_flag%2Cqartod_density_spike_flag%2Cqartod_monotonic_pressure_flag%2Cqartod_pressure_flat_line_flag%2Cqartod_pressure_gross_range_flag%2Cqartod_pressure_primary_flag%2Cqartod_pressure_rate_of_change_flag%2Cqartod_pressure_spike_flag%2Cqartod_salinity_flat_line_flag%2Cqartod_salinity_gross_range_flag%2Cqartod_salinity_primary_flag%2Cqartod_salinity_rate_of_change_flag%2Cqartod_salinity_spike_flag%2Cqartod_temperature_flat_line_flag%2Cqartod_temperature_gross_range_flag%2Cqartod_temperature_primary_flag%2Cqartod_temperature_rate_of_change_flag%2Cqartod_temperature_spike_flag%2Csalinity%2Csalinity_qc%2Ctemperature%2Ctemperature_qc%2Ctime_qc%2Ctime_uv%2Ctime_uv_qc%2Cu%2Cu_qc%2Cv%2Cv_qc&time%3E=2015-08-12T00%3A00%3A00Z&time%3C=2015-08-19T22%3A10%3A22Z)


In [ ]:
ds = open_url("https://gliders.ioos.us/erddap/tabledap/Murphy-20150809T1355")['s']
ds.tree()

Donde las variables `s` identifican los datos secuenciales.


In [ ]:
type(ds)

Podemos identificar más cada dato individual de glider mirando `profile_id`, un valor único para cada uno. Puedes inspeccionar los valores crudos de la siguiente manera.


In [ ]:
[id for id in ds['profile_id'].iterdata()]

```{note}
`OPeNDAP` y por lo tanto `PyDAP` soportan nombres completamente calificados, que son útiles para estructuras de datos complejas (anidadas). En el caso de datos `Sequential`, puedes acceder a una variable `<VarName>` dentro de una secuencia llamada `<Seq>` como `<Seq.VarName>`.
```


Estos datasets son ricos en metadatos, a los que se puede acceder mediante la propiedad attributes de la siguiente manera.


In [ ]:
ds['profile_id'].attributes

Lo primero que queremos hacer es limitar nuestro análisis muy simple. Consideramos solo un glider y solo inspeccionamos las variables `depth` y `temperature`. Esto es similar a una `selección por columnas`.


Para lograrlo usamos la lógica simple de pydap como sigue:


In [ ]:
seq = ds[('profile_id', 'depth', 'temperature')]
print(type(seq))
seq.tree()

### Filtrar datos

Con datos `Sequential` podemos usar `filters` para extraer solo aquellos valores asociados con una o más `rows`. Es decir, identificar todos los valores de la secuencia que son menores, iguales o mayores que cierto valor.

Por ejemplo, consideremos valores de `depth` y `temperature` asociados con el valor `profile_id=5`.


In [ ]:
glid5 = seq[('profile_id', 'depth', 'temperature')][seq['profile_id'].data==5]
glid5

In [ ]:
Depths5 = np.array([depth for depth in glid5['depth']])
ids5 = np.array([_id for _id in glid5['profile_id']])
Temps5 = np.array([temp for temp in glid5['temperature']])

Intentemos otro `profile_id`.


In [ ]:
glid6 = seq[('profile_id', 'depth', 'temperature')][seq['profile_id'].data==6]
Depths6 = np.array([depth for depth in glid6['depth']])
ids6 = np.array([_id for _id in glid6['profile_id']])
Temps6 = np.array([temp for temp in glid6['temperature']])

In [ ]:
plt.plot(Temps5, -Depths5, ls='', marker='o', markersize=10, alpha=0.5, label='profile id: ' +str(ids5[0]))
plt.plot(Temps6, -Depths6, 'k', ls='', marker='d', markersize=10, alpha=0.25, label='profile id: ' +str(ids6[0]))
plt.xlabel(r'$\text{Temp} \;[^\circ C]$', fontsize=12.5)
plt.ylabel(r'$Z\;[m]$', fontsize=12.5, rotation=0, labelpad=10)
plt.legend(fontsize=12.5, frameon=False);

```{warning}
Todas las funcionalidades avanzadas descritas abajo pueden estar desactualizadas.
```

### Funcionalidades avanzadas

**Llamar funciones del lado del servidor**

Cuando abres un dataset remoto, el objeto `DatasetType` tiene un atributo especial llamado `functions` que puede usarse para invocar cualquier función del lado del servidor. Aquí hay un ejemplo de uso de la función `geogrid` de Hyrax:

```{code}
>>> dataset = open_url("http://test.opendap.org/dap/data/nc/coads_climatology.nc")
>>> new_dataset = dataset.functions.geogrid(dataset.SST, 10, 20, -10,60)
>>> new_dataset.SST.COADSY[:] 
[-11. -9. -7. -5. -3. -1. 1. 3. 5. 7. 9. 11.]
>>> new_dataset.SST.COADSX[:]
[ 21. 23. 25. 27. 29. 31. 33. 35. 37. 39. 41. 43. 45. 47. 49. 51. 53. 55. 57. 59. 61.]
```

Desafortunadamente, actualmente no hay un mecanismo estándar para descubrir qué funciones soporta el servidor. El atributo `function` aceptará cualquier nombre de función que la persona usuaria especifique e intentará pasar la llamada al servidor remoto.



### Configurar un proxy
```{warning}
Esto puede estar desactualizado.
```
Es posible configurar pydap para acceder a la red mediante un servidor proxy. Aquí hay un ejemplo para un proxy HTTP ejecutándose en `localhost` y escuchando en el puerto 8000:

```{code}
import httplib2
from pydap.util import socks
import pydap.lib
pydap.lib.PROXY = httplib2.ProxyInfo(
         socks.PROXY_TYPE_HTTP, 'localhost', 8000)
```

De esta forma, todas las llamadas posteriores a `pydap.client.open_url` se enrutarán a través del servidor proxy. También puedes autenticarte en el proxy:

```{code}
pydap.lib.PROXY = httplib2.ProxyInfo(
         socks.PROXY_TYPE_HTTP, 'localhost', 8000,
         proxy_user=USERNAME, proxy_pass=PASSWORD)
```

Una persona usuaria [ha reportado](http://groups.google.com/group/pydap/browse_thread/thread/425b2e1a3b3f233d) que `httplib2` tiene problemas para autenticarse contra un servidor proxy NTLM. En este caso, una solución simple es cambiar la función `pydap.http.request` para que use `urllib2` en lugar de `httplib2`, monkeypatcheando el código como en el [ejemplo de autenticación CAS anterior](#cas):

```{code}
import urllib2
import logging

def install_urllib2_client():
    def new_request(url):
        log = logging.getLogger('pydap')
        log.INFO('Opening %s' % url)

        f = urllib2.urlopen(url.rstrip('?&'))
        headers = dict(f.info().items())
        body = f.read()
        return headers, body

    from pydap.util import http
    http.request = new_request
```

La función `install_urllib2_client` debe llamarse antes de hacer cualquier solicitud.
